# Síntese técnica autônoma de Produtos químicos — v3

O pacote local e os embeddings permanecem prontos. A redação e a auditoria produzem um texto didático para futura geração de QA, com referências e proveniência em arquivos auxiliares. As chamadas Azure rodam no servidor. Leia [DOCUMENTO_DISTRIBUIDO.md](DOCUMENTO_DISTRIBUIDO.md).

In [ ]:
from pathlib import Path
import json
import os
import sys

# Funciona ao abrir o notebook na raiz ou em pipeline/qa_research.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "azure_openai_backend.py").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
BASE = ROOT / "pipeline/qa_research"
PACKET = BASE / "pacotes/produtos_quimicos_v2"
RETURNED = BASE / "relatorios/produtos_quimicos_v3"
print("Projeto:", ROOT)

## Conferir o pacote já preparado

Verifica hashes e limites dos trechos sem carregar modelos. As contagens representam passagens, não necessariamente documentos ou estudos independentes.

In [ ]:
from pipeline.qa_research.document.common import Packet
packet = Packet(PACKET)
print(json.dumps(packet.preparation, ensure_ascii=False, indent=2))
for facet in packet.facets:
    print(f"{facet['id']}: {len(facet['fontes'])} passagens — {facet['titulo']}")

## Inspecionar textos reais antes da execução

Os números apagados permanecem no original. Não serão reconstruídos na síntese. Troque o recorte e o índice para examinar as fontes e os intervalos que o modelo verá.

In [ ]:
facet = packet.facets[0]  # Troque para outro dos 12 recortes.
source_id = facet["fontes"][0]
print("Recorte:", facet["titulo"])
print(json.dumps(packet.views([source_id]), ensure_ascii=False, indent=2))

## Comando para o servidor

O pacote acompanha o Git. Não execute a preparação local lá. Instale `requirements-server.txt` no ambiente do servidor e mantenha suas credenciais Azure naquele ambiente.

In [ ]:
print("""git pull --ff-only origin codex/qa-evidencias-verificadas
bash pipeline/qa_research/rodar_documento_v3.sh""")

## Devolver os resultados

Após terminar no servidor, execute os comandos abaixo lá. Depois execute `git pull --ff-only origin codex/qa-evidencias-verificadas` nesta máquina.

In [ ]:
print(""".venv-documento/bin/python -m pipeline.qa_research.document collect --packet pipeline/qa_research/pacotes/produtos_quimicos_v2 --output pipeline/qa_research/saida_documento_produtos_quimicos_v3 --destination pipeline/qa_research/relatorios/produtos_quimicos_v3
git add -- pipeline/qa_research/relatorios/produtos_quimicos_v3
git commit -m "Resultados do documento Produtos quimicos - sintese v3"
git push origin codex/qa-evidencias-verificadas""")

## Conferir o retorno e ler o documento

O texto didático está em documento.md e documento.json. Referências, origens e validações complementares estão em rastreabilidade.json; cobertura e lacunas são auxiliares. O aviso de término no servidor apresenta o estado e os comandos de retorno.

In [ ]:
from IPython.display import Markdown, display
from pipeline.qa_research.document.common import file_hash

if RETURNED.is_dir():
    manifest = json.loads((RETURNED / "manifest_resultados.json").read_text(encoding="utf-8"))
    changed = [name for name, digest in manifest["files"].items()
               if not (RETURNED / name).is_file() or file_hash(RETURNED / name) != digest]
    if changed:
        raise ValueError(f"Arquivos ausentes/alterados: {changed}")
    summary = json.loads((RETURNED / "resumo.json").read_text(encoding="utf-8"))
    coverage = json.loads((RETURNED / "cobertura.json").read_text(encoding="utf-8"))
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    for row in coverage:
        print(row["id"], row["status"], "blocos:", row["blocos_publicados"], "lacunas:", row["lacunas"])
    display(Markdown((RETURNED / "documento.md").read_text(encoding="utf-8")))
else:
    print("Resultados ainda não devolvidos pelo servidor.")

## Preparar uma nova rodada local (opcional)

Execute somente se precisar ampliar as fontes. O pacote v2 já está pronto e continua sendo usado na síntese v3. O cache local evita repetir embeddings de textos iguais. Um feedback novo pode exigir nova leitura dos originais. A preparação não faz chamadas Azure.

In [ ]:
PREPARAR_NOVO_PACOTE = False
if PREPARAR_NOVO_PACOTE:
    from pipeline.qa_research.document.prepare import prepare
    feedback = RETURNED / "pedidos_busca.json"
    info = prepare(
        BASE / "configs/documento_produtos_quimicos.json",
        BASE / "pacotes/produtos_quimicos_proxima_rodada",
        feedback=feedback if feedback.is_file() else None,
    )
    print(json.dumps(info, ensure_ascii=False, indent=2))